# milestone 5model ensemble and test-time augmentation

In [ ]:
# setup - load models and data
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# fine-tuned deberta
deberta_tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')
deberta_model = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-small', num_labels=5)
deberta_model.to(device)
deberta_model.eval()

# fine-tuned roberta
roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
roberta_model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=5)
roberta_model.to(device)
roberta_model.eval()

# load data
test = pd.read_csv('../data/test.csv')
train = pd.read_csv('../data/train.csv')

options_list = ['A', 'B', 'C', 'D', 'E']
label_to_option = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

def predict_probs(model, tokenizer, row):
    text = f"{row['prompt']} [SEP] {row['A']} [SEP] {row['B']} [SEP] {row['C']} [SEP] {row['D']} [SEP] {row['E']}"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    return F.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

print('models and data loaded successfully')

Loading weights: 100|██████████| 199/199 [00:00<00:00, 1000it/s]


models and data loaded successfully


In [ ]:
# q1 - deberta highest prob for row 25
row_25 = test.iloc[24]
probs_d = predict_probs(deberta_model, deberta_tokenizer, row_25)
top_idx = np.argmax(probs_d)
print(f"{label_to_option[top_idx]}, {probs_d[top_idx]}")

A, 0.29296875


In [ ]:
# q2 - highest averaged prob for row 25
probs_r = predict_probs(roberta_model, roberta_tokenizer, row_25)
avg_probs = (probs_d + probs_r) / 2
top_avg = label_to_option[np.argmax(avg_probs)]
print(top_avg)

A


In [ ]:
# q3 - weighted ensemble best option
weighted_probs = 0.7 * probs_d + 0.3 * probs_r
top_weighted = label_to_option[np.argmax(weighted_probs)]
print(top_weighted)

A


In [ ]:
# q4 - top-3 prediction string for row 25
top3_idxs = np.argsort(weighted_probs)[::-1][:3]
top3_str = ' '.join([label_to_option[i] for i in top3_idxs])
print(top3_str)

A D E


In [ ]:
# q5 - generate submission.csv and count rows
predictions = []
for idx, row in test.iterrows():
    probs_d = predict_probs(deberta_model, deberta_tokenizer, row)
    probs_r = predict_probs(roberta_model, roberta_tokenizer, row)
    weighted = 0.7 * probs_d + 0.3 * probs_r
    top3 = ' '.join([label_to_option[i] for i in np.argsort(weighted)[::-1][:3]])
    predictions.append({'id': row['id'], 'prediction': top3})

sub_df = pd.DataFrame(predictions)
sub_df.to_csv('../submission.csv', index=False)
print(len(sub_df))

500


In [ ]:
# q6 - tta on first 50 rows of test.csv
tta_diff = 0
for idx, row in test.iloc[:50].iterrows():
    probs_orig = predict_probs(deberta_model, deberta_tokenizer, row)

    row_aug = row.copy()
    row_aug['prompt'] = "Answer the following multiple-choice question carefully: " + str(row['prompt'])
    probs_aug = predict_probs(deberta_model, deberta_tokenizer, row_aug)

    probs_avg = (probs_orig + probs_aug) / 2

    if np.argmax(probs_orig) != np.argmax(probs_avg):
        tta_diff += 1

print(tta_diff)

3


In [ ]:
# q7 - compare deberta vs ensemble top-1 for first 100 test rows
diff_top1 = 0
for idx, row in test.iloc[:100].iterrows():
    probs_d = predict_probs(deberta_model, deberta_tokenizer, row)
    probs_r = predict_probs(roberta_model, roberta_tokenizer, row)
    weighted = 0.7 * probs_d + 0.3 * probs_r

    if np.argmax(probs_d) != np.argmax(weighted):
        diff_top1 += 1

print(diff_top1)

5


In [ ]:
# q8 - confidence gain for first 100 test rows
positive_gain = 0
for idx, row in test.iloc[:100].iterrows():
    probs_d = predict_probs(deberta_model, deberta_tokenizer, row)
    probs_r = predict_probs(roberta_model, roberta_tokenizer, row)
    weighted = 0.7 * probs_d + 0.3 * probs_r

    conf_d = np.max(probs_d)
    conf_ens = np.max(weighted)

    if conf_ens - conf_d > 0:
        positive_gain += 1

print(positive_gain)

0


In [ ]:
# q9 - top-3 ranking changes after ensembling
changes = 0
for idx, row in test.iloc[:100].iterrows():
    probs_d = predict_probs(deberta_model, deberta_tokenizer, row)
    probs_r = predict_probs(roberta_model, roberta_tokenizer, row)
    weighted = 0.7 * probs_d + 0.3 * probs_r

    top3_d = list(np.argsort(probs_d)[::-1][:3])
    top3_ens = list(np.argsort(weighted)[::-1][:3])

    if top3_d != top3_ens:
        changes += 1

print(changes)

23


In [ ]:
# q10 - map@3 on first 100 train rows
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score

ap_scores = []
for idx, row in train.iloc[:100].iterrows():
    probs_d = predict_probs(deberta_model, deberta_tokenizer, row)
    probs_r = predict_probs(roberta_model, roberta_tokenizer, row)
    weighted = 0.7 * probs_d + 0.3 * probs_r

    top3 = [label_to_option[i] for i in np.argsort(weighted)[::-1][:3]]
    ap_scores.append(apk(row['answer'], top3, k=3))

print(round(np.mean(ap_scores), 4))

0.31
